# SingleBehaviorLab — Colab demo (segmentation & unsupervised discovery)

Run the full segmentation → embedding → clustering pipeline directly in Google Colab on a bundled 3-minute silent demo video.

**Before you run:**

1. Select a GPU runtime via **Runtime → Change runtime type → GPU** (any of the T4 / L4 / A100 tiers works).
2. The first install cell pulls torch, jax, and the vendored SAM2/VideoPrism packages. Expect **5–10 minutes on the first run**.
3. Colab may prompt you to **restart the runtime** after installing. If that happens, click *Restart* and re-run the cells from the top — the subsequent installs are cached.
4. Notebook runs end-to-end on the default 2000-frame subset in roughly 3–5 minutes post-install. Flip `FULL_VIDEO = True` in the configuration cell to process the whole clip.

## 1. Install SingleBehaviorLab

In [ ]:
%pip install --quiet singlebehaviorlab


## 2. Verify the GPU runtime

Both PyTorch and JAX should report a CUDA device. If this cell shows CPU-only, switch the runtime via *Runtime → Change runtime type* and re-run.

In [ ]:
import torch, jax

print(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()}")
print(f"jax   {jax.__version__} | devices: {jax.devices()}")
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime → Change runtime type → GPU"

## 3. Download the demo dataset

`sbl.load_demo` fetches the bundled video and prompts file into Colab's local disk and returns their paths. Subsequent calls reuse the cached copies.

In [ ]:
import singlebehaviorlab as sbl

demo = sbl.load_demo("segmentation_clustering")
VIDEO = demo["video"]
PROMPTS = demo["prompts"]
print("Video:  ", VIDEO)
print("Prompts:", PROMPTS)

## 4. Configuration

Flip `FULL_VIDEO = True` to process the whole clip. The default (`False`, 2000 frames) runs end-to-end in a few minutes.

In [ ]:
from pathlib import Path

WORK_DIR = Path("/content/sbl_demo")
WORK_DIR.mkdir(parents=True, exist_ok=True)

FULL_VIDEO = False
END_FRAME = None if FULL_VIDEO else 2000
SAM2_MODEL = "sam2.1_hiera_large.pt"

STEM = Path(VIDEO).stem + ("_full" if FULL_VIDEO else f"_{END_FRAME}")
MASKS = WORK_DIR / f"{STEM}_masks.h5"
MATRIX = WORK_DIR / f"{STEM}_matrix.npz"
ANALYSIS = WORK_DIR / f"{STEM}_analysis.pkl"
PLOT_PDF = WORK_DIR / f"{STEM}_umap.pdf"

print(f"Work dir: {WORK_DIR}")
print(f"Mode:     {'full video' if FULL_VIDEO else f'first {END_FRAME} frames'}")

## 5. Segment with SAM2

`sbl.segment` loads the bundled SAM2 checkpoint, applies the point prompts, and propagates masks across the requested frame range in 200-frame chunks so it never has to hold the entire video in GPU memory.

In [ ]:
sbl.segment(
    VIDEO,
    PROMPTS,
    MASKS,
    model_name=SAM2_MODEL,
    start_frame=0,
    end_frame=END_FRAME,
    log_fn=print,
)
print(f"\nMask file: {MASKS}")

## 6. Extract VideoPrism embeddings

In [ ]:
params = sbl.RegistrationParams(target_fps=25, clip_length_frames=16)

reg_result = sbl.register(
    VIDEO,
    MASKS,
    MATRIX,
    params=params,
    log_fn=print,
)
print(f"\nMatrix:   {reg_result['matrix']}")
print(f"Metadata: {reg_result['metadata']}")

## 7. Cluster and plot the UMAP inline

In [ ]:
cluster_params = sbl.ClusteringParams(
    method="leiden",
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    leiden_resolution=1.0,
)

sbl.cluster(
    reg_result["matrix"],
    ANALYSIS,
    metadata_path=reg_result["metadata"],
    params=cluster_params,
    log_fn=print,
)

fig = sbl.plot_umap_clusters(
    ANALYSIS,
    show=False,
    save=PLOT_PDF,
    title=f"UMAP — {STEM}",
    point_size=10,
)
fig

## 8. Next steps

- The files under `/content/sbl_demo/` are ephemeral — download `PLOT_PDF` and `ANALYSIS` to your local machine if you want to keep them.
- Open the saved analysis `.pkl` in the desktop GUI (**Clustering tab → Load Analysis State**) to browse clusters interactively and click UMAP points to see the underlying clips.
- To try the pipeline on your own video, upload it to the Colab runtime, export point prompts from the desktop GUI via **Segmentation tab → Export prompts**, and point `VIDEO` and `PROMPTS` at those files instead of calling `sbl.load_demo`.
- Full CLI reference and headless server workflow: see [`CLI.md`](https://github.com/alms93/SingleBehaviorLab/blob/main/CLI.md) in the repository.